In [0]:
from datetime import datetime 

In [0]:
## widgets

dbutils.widgets.text("batch_id" , " 1" , "Batch ID (1,2 or 3)")

In [0]:
batch_id = dbutils.widgets.get("batch_id").strip()
run_id = datetime.now().strftime("%Y%m%d%H%M%S")

print(f"batch_id: {batch_id}")
print(f"run_id: {run_id}")

In [0]:
team_name = "team_lemma"
source_base = "abfss://raw@schwabdldevsa.dfs.core.windows.net"
catalog     = f"charles_schwab_retailbrokerage_dev_{team_name}"
landing_volume = f"/Volumes/{catalog}/landing/pwg"
     
Batch_Folder = f"Batch{batch_id}"
Source_Path = f"{source_base}/{Batch_Folder}/DailyMarket.txt"
Target_Path = f"{landing_volume}/{Batch_Folder}/dailymarket"

In [0]:
## batch 1 columns
Dailymarket_B1_Cols = ["DM_DATE" , "DM_S_SYMB" , "DM_CLOSE" , "DM_HIGH" , "DM_LOW" , "DM_VOL"]

## batch2/3 columns

Dailymarket_B23_Cols = ["DM_ACTION" , "DM_RECID", "DM_DATE","DM_S_SYMB" , "DM_CLOSE" , "DM_HIGH" , "DM_LOW" , "DM_VOL"]

if batch_id == "1":
    column_names = Dailymarket_B1_Cols
else:
    print("batch23")
    column_names = Dailymarket_B23_Cols

print(f"For Batch {batch_id} columns are {column_names}")




In [0]:
## read data from adls

df_raw = spark.read.option("header" , "false")\
                   .option("sep" , "|")\
                   .option("inferSchema", False)\
                   .csv(Source_Path)

df_named = df_raw.toDF(*column_names)

print(f"row count is for batch{batch_id} : {df_named.count()}")
df_named.printSchema()
                   

## normalize schema

In [0]:
from pyspark.sql.functions import lit , current_timestamp

In [0]:
## Normalize B1 schema : 
if batch_id == "1":
    df_named = df_named.withColumn("DM_ACTION" , lit("I"))\
                    .withColumn("DM_RECID" , lit(None).cast("String"))

    df_named = df_named.select("DM_ACTION","DM_RECID" , "DM_DATE" ,"DM_S_SYMB" , "DM_CLOSE" , "DM_HIGH" , "DM_LOW" , "DM_VOL")
    
    print("B1 normalized:DM_ACTION='I', DM_RECID=NULL added ")


print("final schema")

df_named.printSchema()

    



In [0]:
## adding metadata column

df_landing = df_named\
                 .withColumn("_source_file" , lit("DailyMarket.txt"))\
                .withColumn("_batch" , lit(batch_id))\
                .withColumn("_run_id" , lit(run_id))\
                .withColumn("_ingest_ts" , current_timestamp())

# df_landing.printSchema()
print(df_landing.columns)

### write to the Landing volume

In [0]:
df_landing.write.mode("overwrite").parquet(Target_Path)
landing_count = spark.read.parquet(Target_Path).count()
source_count  = df_landing.count()

print(f"Source rows  : {source_count}")
print(f"Landing rows : {landing_count}")
print(f"Status       : {'MATCH' if source_count == landing_count else 'MISMATCH'}")
print(f"Target path  : {Target_Path}")


In [0]:
EXPECTED = {"1" : 5270304 , "2" : 7360 , "3" : 7360}
expected = EXPECTED[batch_id]

if landing_count == expected:
    print("pass")
else:
    print("fail")


### Audit Log

In [0]:
%run ../../02_common_utils/operations

In [0]:
from pyspark.sql import Row

recon_results = [
    Row(
        source_table = "dailymarket",
        batch_id = Batch_Folder,
        target_path = Target_Path,
        source_count = source_count,
        landing_count = landing_count,
        status = "Match" if source_count==landing_count else "Not Match",
        columns_applied = "raw_line"
    )
]

recon_df = spark.createDataFrame(recon_results)

for row in recon_results:
    if row.status in ("Match" , "Not Match"):
        log_pipeline_recon(
            spark         = spark,
            run_id        = run_id,
            batch_id      = row.batch_id,
            domain        = "MARKET",
            table_name    = row.source_table,
            source_layer  = "raw",
            target_layer  = "landing",
            source_count  = row.source_count,
            target_count  = row.landing_count
        )

        log_audit_event(
            spark = spark,
            run_id = run_id,
            batch = row.batch_id,
            layer = "Landing",
            table_name = row.source_table,
            operation = "Append",
            rows_affected  = row.landing_count
        )

display(recon_df)